# Step 9: Deploy the Model

**SageMaker Unified Studio Component**: Inference Endpoints

In [ ]:
import sagemaker
import os
from sagemaker.sklearn import SKLearnModel
from dotenv import load_dotenv

load_dotenv()
bucket_name = os.getenv('BUCKET_NAME')

# Use SageMaker's execution role (recommended for Unified Studio)
try:
    role = sagemaker.get_execution_role()
    print(f"Using SageMaker execution role: {role}")
except ValueError:
    # Fallback to .env file role if running locally
    role = os.getenv('EXECUTION_ROLE')
    print(f"Using .env execution role: {role}")

## Deploy Endpoint

In [ ]:
model_data = f's3://{bucket_name}/models/logistic_regression/model.tar.gz'

sklearn_model = SKLearnModel(
    model_data=model_data,
    role=role,
    entry_point='inference.py',
    framework_version='1.2-1',
    py_version='py3'
)

predictor = sklearn_model.deploy(
    initial_instance_count=1,
    instance_type='ml.t2.medium',
    endpoint_name='machine-overheat-endpoint'
)

print(f"✓ Endpoint deployed: {predictor.endpoint_name}")

## Test the Endpoint

In [ ]:
from sagemaker.predictor import Predictor
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

# Connect to existing endpoint (or use predictor from deploy cell)
try:
    # If predictor exists from deployment
    predictor.serializer = JSONSerializer()
    predictor.deserializer = JSONDeserializer()
except NameError:
    # Connect to existing endpoint
    predictor = Predictor(
        endpoint_name='machine-overheat-endpoint',
        serializer=JSONSerializer(),
        deserializer=JSONDeserializer()
    )
    print("Connected to existing endpoint: machine-overheat-endpoint")

test_input = {'temperature': 78, 'room_temp': 25}
response = predictor.predict(test_input)
print(f"Input: {test_input}")
print(f"Prediction: {response}")

In [ ]:
test_input = {'temperature': 85, 'room_temp': 25}
response = predictor.predict(test_input)
print(f"Input: {test_input}")
print(f"Prediction: {response}")

## Cleanup (Optional)

In [ ]:
# predictor.delete_endpoint()
# print("Endpoint deleted")